## 17B. EXTENDED ABLATION STUDY AND EDGE CASES

The single-split ablation above (Section 17) uses one 61-patient hold-out, no uncertainty estimates
and preprocessing fitted on all 303 rows. Following the reviewers' request (action-plan row 16),
this section re-tests every claim under **one controlled protocol**:

| Item | Setting |
|---|---|
| Data | Raw UCI Cleveland files; missing values kept and imputed **inside** each training fold |
| Splits | Repeated stratified 10-fold CV (5 × 10 = 50 folds), identical for every variant |
| Pipeline | median imputation → ECG-proxy features → StandardScaler → resampling (train only) → classifier |
| Hyper-parameters | Fixed Section-15 values for every variant (identical tuning budget) |
| Metrics | AUC, accuracy, sensitivity, specificity, F1, Brier score, calibration slope |
| Uncertainty | 95% CI from the Nadeau–Bengio corrected resampled t-test; paired ΔAUC; Holm correction |

Sub-sections: 17B-1 component ablation · 17B-2 control experiments and permutation test ·
17B-3 imbalance strategies · 17B-4 protocol and leakage · 17B-5 robustness and edge cases ·
17B-6 encoding and calibration · 17B-7 external validation on three other hospitals ·
17B-8 SHAP stability · 17B-9 cross-attention · 17B-10 PTB-XL ECG branch · 17B-11 verdict.

Results are written to `outputs/ablation/` (set `MCF_ABL_DIR` to change); completed runs are cached,
so re-running a cell is fast. Set `MCF_REPEATS=3` for a quicker run.


In [ ]:
# ══════════════════════════════════════════════════════════════════════════
# 17B. EXTENDED ABLATION STUDY — shared setup
# ──────────────────────────────────────────────────────────────────────────
# Every experiment in this section uses the SAME protocol so that variants are
# directly comparable (reviewer action-plan row 16):
#   • raw Cleveland data (missing values kept) → all preprocessing happens
#     INSIDE each training fold: median imputation → ECG-proxy features →
#     StandardScaler → resampling (training rows only) → classifier
#   • repeated stratified 10-fold CV (N_REPEATS × 10 folds, identical splits
#     for every variant)
#   • identical, fixed hyper-parameters for every variant (the values found by
#     RandomizedSearchCV in Section 15), i.e. the same tuning budget
#   • metrics per fold: AUC, accuracy, sensitivity, specificity, F1, Brier
#   • uncertainty: 95% CI from the Nadeau–Bengio corrected resampled t-test,
#     which accounts for the overlap between CV training sets; differences
#     between variants are paired by fold; p-values are Holm-corrected
# ══════════════════════════════════════════════════════════════════════════
import os, time, json, pickle, hashlib, warnings
import numpy as np
import pandas as pd
from scipy import stats
import matplotlib.pyplot as plt
from sklearn.base import BaseEstimator, TransformerMixin, ClassifierMixin, clone
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import RepeatedStratifiedKFold, StratifiedKFold, train_test_split
from sklearn.metrics import roc_auc_score, f1_score, brier_score_loss, confusion_matrix
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.ensemble import (RandomForestClassifier, GradientBoostingClassifier,
                              VotingClassifier, StackingClassifier)
from xgboost import XGBClassifier
from imblearn.pipeline import Pipeline as ImbPipeline
from imblearn.over_sampling import (SMOTE, SMOTENC, ADASYN, BorderlineSMOTE,
                                    RandomOverSampler)
from joblib import Parallel, delayed

warnings.filterwarnings('ignore')
os.environ['PYTHONWARNINGS'] = 'ignore'          # also silences joblib worker processes
os.environ.setdefault('TF_CPP_MIN_LOG_LEVEL', '3')

SEED      = 42
N_FOLDS   = 10
N_REPEATS = int(os.environ.get('MCF_REPEATS', 5))      # 5 × 10 = 50 folds
N_JOBS    = int(os.environ.get('MCF_JOBS', -1))
ABL_DIR   = os.environ.get('MCF_ABL_DIR', 'outputs/ablation')
FIG_DIR, TAB_DIR = f'{ABL_DIR}/figures', f'{ABL_DIR}/tables'
for d in (FIG_DIR, TAB_DIR, f'{ABL_DIR}/cache'):
    os.makedirs(d, exist_ok=True)

# ── Data: raw UCI files (missing values kept as NaN) ──────────────────────
CLIN_COLS = ['age', 'sex', 'cp', 'trestbps', 'chol', 'fbs', 'restecg',
             'thalach', 'exang', 'oldpeak', 'slope', 'ca', 'thal']
CAT_COLS  = ['sex', 'cp', 'fbs', 'restecg', 'exang', 'slope', 'ca', 'thal']
CONT_COLS = ['age', 'trestbps', 'chol', 'thalach', 'oldpeak']
UCI_URL   = 'https://archive.ics.uci.edu/ml/machine-learning-databases/heart-disease/'

def load_uci_site(site, chol_zero_as_missing=True):
    """Load one site of the UCI heart-disease data. Target: num > 0 → 1."""
    for root in ('data/uci', '../data/uci'):
        path = f'{root}/processed.{site}.data'
        if os.path.exists(path):
            break
    else:
        path = UCI_URL + f'processed.{site}.data'
    df = pd.read_csv(path, header=None, names=CLIN_COLS + ['num'], na_values='?')
    y = (df.pop('num') > 0).astype(int).values
    df = df.astype(float)
    if chol_zero_as_missing:          # Switzerland/VA code missing cholesterol as 0
        df.loc[df['chol'] == 0, 'chol'] = np.nan
    return df, y

Xdf_cl, y_cl = load_uci_site('cleveland')
X_cl_raw = Xdf_cl.values
print(f"Cleveland: {X_cl_raw.shape}, healthy={np.sum(y_cl == 0)} "
      f"({np.mean(y_cl == 0):.1%}), disease={np.sum(y_cl == 1)} ({np.mean(y_cl == 1):.1%}), "
      f"missing cells={int(np.isnan(X_cl_raw).sum())}")

# ── ECG-proxy features (identical formulas to Section 9) ──────────────────
PROXY_GROUPS = {
    'raw':         ['restecg', 'oldpeak', 'slope', 'thalach', 'exang'],
    'polynomial':  ['oldpeak_sq', 'thalach_sq'],
    'interaction': ['oldpeak_x_slope', 'thalach_x_exang', 'oldpeak_x_thalach',
                    'restecg_x_oldpeak', 'slope_x_thalach'],
    'log':         ['log_oldpeak', 'log_thalach'],
    'ratio':       ['oldpeak_per_thalach', 'thalach_per_slope'],
    'threshold':   ['high_oldpeak', 'low_thalach', 'abnormal_restecg', 'downslope'],
    'hr_reserve':  ['hr_reserve', 'hr_reserve_pct'],
    'composite':   ['ischaemia_score'],
}
ALL_GROUPS = tuple(PROXY_GROUPS)

def ecg_proxy_frame(d, fix_downslope=False):
    """23 ECG-proxy columns. fix_downslope=True uses the correct UCI code (3)."""
    e = pd.DataFrame(index=d.index)
    for c in PROXY_GROUPS['raw']:
        e[c] = d[c]
    op_s = np.clip(d['oldpeak'], 0, None); th_s = np.clip(d['thalach'], 1, None)
    e['oldpeak_sq'] = d['oldpeak'] ** 2;            e['thalach_sq'] = d['thalach'] ** 2
    e['oldpeak_x_slope'] = d['oldpeak'] * d['slope']
    e['thalach_x_exang'] = d['thalach'] * d['exang']
    e['oldpeak_x_thalach'] = d['oldpeak'] * d['thalach']
    e['restecg_x_oldpeak'] = d['restecg'] * d['oldpeak']
    e['slope_x_thalach'] = d['slope'] * d['thalach']
    e['log_oldpeak'] = np.log1p(op_s);              e['log_thalach'] = np.log(th_s)
    e['oldpeak_per_thalach'] = op_s / th_s
    e['thalach_per_slope'] = th_s / np.clip(d['slope'], 1, None)
    e['high_oldpeak'] = (d['oldpeak'] > 1.0).astype(int)
    e['low_thalach'] = (d['thalach'] < 140).astype(int)
    e['abnormal_restecg'] = (d['restecg'] > 0).astype(int)
    e['downslope'] = (d['slope'] == (3 if fix_downslope else 2)).astype(int)
    pred_max = np.clip(220 - np.clip(d['age'], 1, None), 1, None)
    e['hr_reserve'] = th_s - pred_max;              e['hr_reserve_pct'] = e['hr_reserve'] / pred_max
    e['ischaemia_score'] = op_s * (d['slope'] + 1) + d['exang'] * 2
    return e

NONECG_NAMES = ['age_sq', 'chol_sq', 'trestbps_sq', 'age_x_chol', 'age_x_trestbps',
                'chol_x_trestbps', 'sex_x_age', 'fbs_x_chol', 'log_chol', 'log_trestbps',
                'chol_per_age', 'trestbps_per_age', 'high_chol', 'high_bp', 'old_age',
                'cp_x_age', 'ca_x_age', 'thal_x_sex']

def nonecg_control_frame(d):
    """18 engineered NON-ECG features (same transform types as the proxy block).
    Control for 'does adding ANY engineered features help?'"""
    e = pd.DataFrame(index=d.index)
    e['age_sq'] = d['age'] ** 2; e['chol_sq'] = d['chol'] ** 2; e['trestbps_sq'] = d['trestbps'] ** 2
    e['age_x_chol'] = d['age'] * d['chol']; e['age_x_trestbps'] = d['age'] * d['trestbps']
    e['chol_x_trestbps'] = d['chol'] * d['trestbps']; e['sex_x_age'] = d['sex'] * d['age']
    e['fbs_x_chol'] = d['fbs'] * d['chol']
    e['log_chol'] = np.log(np.clip(d['chol'], 1, None)); e['log_trestbps'] = np.log(np.clip(d['trestbps'], 1, None))
    e['chol_per_age'] = d['chol'] / np.clip(d['age'], 1, None)
    e['trestbps_per_age'] = d['trestbps'] / np.clip(d['age'], 1, None)
    e['high_chol'] = (d['chol'] > 240).astype(int); e['high_bp'] = (d['trestbps'] > 140).astype(int)
    e['old_age'] = (d['age'] >= 55).astype(int); e['cp_x_age'] = d['cp'] * d['age']
    e['ca_x_age'] = d['ca'] * d['age']; e['thal_x_sex'] = d['thal'] * d['sex']
    return e

ONEHOT_LEVELS = {'cp': [1, 2, 3, 4], 'restecg': [0, 1, 2], 'slope': [1, 2, 3], 'thal': [3, 6, 7]}

class ProxyFeatures(BaseEstimator, TransformerMixin):
    """Fold-safe feature builder working on the (imputed) 13 clinical columns.

    groups        : ECG-proxy groups to append (empty tuple → clinical only)
    keep_clinical : keep the 13 clinical columns
    drop_dup_raw  : drop proxy 'raw' columns that duplicate clinical ones
    extra         : None | 'noise' | 'nonecg' | 'shuffled_proxy'  (control blocks)
    onehot        : one-hot encode cp/restecg/slope/thal instead of ordinal codes
    """
    def __init__(self, groups=ALL_GROUPS, keep_clinical=True, fix_downslope=False,
                 drop_dup_raw=False, extra=None, onehot=False, seed=SEED):
        self.groups, self.keep_clinical, self.fix_downslope = groups, keep_clinical, fix_downslope
        self.drop_dup_raw, self.extra, self.onehot, self.seed = drop_dup_raw, extra, onehot, seed

    def fit(self, X, y=None):
        return self

    def _cols(self):
        cols = [c for g in self.groups for c in PROXY_GROUPS[g]]
        if self.drop_dup_raw:
            cols = [c for c in cols if c not in PROXY_GROUPS['raw']]
        return cols

    def transform(self, X):
        d = pd.DataFrame(np.asarray(X, dtype=float), columns=CLIN_COLS)
        parts, names = [], []
        if self.keep_clinical:
            if self.onehot:
                ordinal = [c for c in CLIN_COLS if c not in ONEHOT_LEVELS]
                parts.append(d[ordinal].values); names += ordinal
                for c, lv in ONEHOT_LEVELS.items():
                    for v in lv:
                        parts.append((np.round(d[c].values) == v).astype(float)[:, None])
                        names.append(f'{c}={v}')
            else:
                parts.append(d.values); names += CLIN_COLS
        cols = self._cols()
        if cols:
            P = ecg_proxy_frame(d, self.fix_downslope)[cols].values
            if self.extra == 'shuffled_proxy':     # break the row ↔ patient link
                rng = np.random.RandomState(self.seed + len(d))
                P = P[rng.permutation(len(d))]
            parts.append(P); names += [f'proxy:{c}' if c in CLIN_COLS else c for c in cols]
        if self.extra == 'nonecg':
            parts.append(nonecg_control_frame(d).values); names += NONECG_NAMES
        elif self.extra == 'noise':
            rng = np.random.RandomState(self.seed + len(d))
            parts.append(rng.randn(len(d), 18)); names += [f'noise_{i}' for i in range(18)]
        self.feature_names_ = names
        return np.nan_to_num(np.hstack(parts), nan=0.0, posinf=0.0, neginf=0.0)

def feature_names(feat):
    f = clone(feat)
    f.fit(None).transform(SimpleImputer(strategy='median').fit_transform(X_cl_raw))
    return f.feature_names_

# ── Models: fixed hyper-parameters from Section 15 (same budget everywhere) ─
TUNED = {
    'xgb': dict(n_estimators=200, max_depth=3, learning_rate=0.01, subsample=1.0,
                colsample_bytree=0.7, min_child_weight=1, gamma=0),
    'svm': dict(kernel='rbf', C=100, gamma=0.001),
    'gbm': dict(n_estimators=200, max_depth=3, learning_rate=0.01, subsample=1.0, max_features=None),
    'rf':  dict(n_estimators=500, max_depth=20, min_samples_split=5, max_features='log2'),
}
POS_WEIGHT = float(np.sum(y_cl == 0) / np.sum(y_cl == 1))

def make_base(name, balanced=False):
    cw = 'balanced' if balanced else None
    if name == 'xgb':
        return XGBClassifier(**TUNED['xgb'], eval_metric='logloss', random_state=SEED,
                             verbosity=0, n_jobs=1,
                             scale_pos_weight=POS_WEIGHT if balanced else 1.0)
    if name == 'svm':
        return SVC(**TUNED['svm'], probability=True, random_state=SEED, class_weight=cw)
    if name == 'gbm':
        return GradientBoostingClassifier(**TUNED['gbm'], random_state=SEED)
    if name == 'rf':
        return RandomForestClassifier(**TUNED['rf'], random_state=SEED, n_jobs=1, class_weight=cw)
    if name == 'lr':
        return LogisticRegression(max_iter=2000, random_state=SEED, class_weight=cw)
    raise ValueError(name)

class HardVoting(BaseEstimator, ClassifierMixin):
    """Hard (majority) voting; the score used for AUC is the fraction of votes."""
    def __init__(self, members=('xgb', 'svm', 'gbm', 'rf')):
        self.members = members
    def fit(self, X, y):
        self.models_ = [make_base(m).fit(X, y) for m in self.members]
        self.classes_ = np.array([0, 1]); return self
    def predict_proba(self, X):
        v = np.mean([m.predict(X) for m in self.models_], axis=0)
        return np.column_stack([1 - v, v])
    def predict(self, X):
        return (self.predict_proba(X)[:, 1] > 0.5).astype(int)

def make_clf(kind='voting', balanced=False):
    members = [(m, make_base(m, balanced)) for m in ('xgb', 'svm', 'gbm', 'rf')]
    if kind == 'voting':
        return VotingClassifier(members, voting='soft')
    if kind == 'hard_voting':
        return HardVoting()
    if kind == 'stacking':
        return StackingClassifier(members, final_estimator=LogisticRegression(max_iter=2000),
                                  cv=5, stack_method='predict_proba')
    return make_base(kind, balanced)

SAMPLERS = {
    None:          lambda: None,
    'smote':       lambda: SMOTE(random_state=SEED),
    'ros':         lambda: RandomOverSampler(random_state=SEED),
    'borderline':  lambda: BorderlineSMOTE(random_state=SEED),
    'adasyn':      lambda: ADASYN(random_state=SEED),
    'smotenc':     lambda: SMOTENC(categorical_features=[CLIN_COLS.index(c) for c in CAT_COLS],
                                   random_state=SEED),
}

def make_pipe(feat=None, sampler='smote', clf='voting', sampler_stage='post', balanced=False,
              impute=True, scale=True):
    """sampler_stage='post' (as in the paper): resample AFTER features+scaling.
       sampler_stage='pre' : resample the raw imputed clinical rows, THEN derive
                             the proxy features (synthetic patients stay internally
                             consistent, e.g. oldpeak_sq == oldpeak**2)."""
    feat = ProxyFeatures() if feat is None else feat
    smp = SAMPLERS[sampler]()
    steps = []
    if impute:
        steps.append(('impute', SimpleImputer(strategy='median', keep_empty_features=True)))
    if smp is not None and sampler_stage == 'pre':
        steps.append(('sampler', smp))
    steps.append(('features', feat))
    if scale:
        steps.append(('scale', StandardScaler()))
    if smp is not None and sampler_stage == 'post':
        steps.append(('sampler', smp))
    steps.append(('clf', make_clf(clf, balanced) if isinstance(clf, str) else clf))
    return ImbPipeline(steps)

FEAT_FULL     = lambda **kw: ProxyFeatures(**kw)
FEAT_CLINICAL = lambda **kw: ProxyFeatures(groups=(), **kw)

# ── Evaluation engine ──────────────────────────────────────────────────────
SPLITS = list(RepeatedStratifiedKFold(n_splits=N_FOLDS, n_repeats=N_REPEATS,
                                      random_state=SEED).split(X_cl_raw, y_cl))
TEST_TRAIN_RATIO = 1 / (N_FOLDS - 1)
RESULTS = {}            # name → dict(folds=DataFrame, oof=array[repeats, n])

def fold_metrics(y, p, thr=0.5):
    yhat = (p >= thr).astype(int)
    tn, fp, fn, tp = confusion_matrix(y, yhat, labels=[0, 1]).ravel()
    return {'AUC': roc_auc_score(y, p) if len(np.unique(y)) > 1 else np.nan,
            'Accuracy': (tp + tn) / len(y),
            'Sensitivity': tp / (tp + fn) if (tp + fn) else np.nan,
            'Specificity': tn / (tn + fp) if (tn + fp) else np.nan,
            'F1': f1_score(y, yhat, zero_division=0),
            'Brier': brier_score_loss(y, p)}

def _fit_predict(pipe, X, y, tr, te, k, train_tf, test_tf):
    Xtr, ytr, Xte = X[tr].copy(), y[tr].copy(), X[te].copy()
    if train_tf is not None:
        Xtr, ytr = train_tf(Xtr, ytr, SEED + k)
    if test_tf is not None:
        Xte = test_tf(Xte, SEED + 10_000 + k)
    m = clone(pipe).fit(Xtr, ytr)
    return m.predict_proba(Xte)[:, 1]

def _cache_key(name, n_splits):
    return f'{ABL_DIR}/cache/' + hashlib.md5(f'{name}|{n_splits}|{N_REPEATS}'.encode()).hexdigest() + '.pkl'

def evaluate(name, pipe, X=None, y=None, splits=None, train_tf=None, test_tf=None,
             n_jobs=None, use_cache=True, verbose=True):
    """Run `pipe` on every split; store per-fold metrics and out-of-fold probabilities."""
    X = X_cl_raw if X is None else X
    y = y_cl if y is None else y
    splits = SPLITS if splits is None else splits
    key = _cache_key(name, len(splits))
    if use_cache and os.path.exists(key):
        RESULTS[name] = pickle.load(open(key, 'rb'))
    else:
        t0 = time.time()
        probs = Parallel(n_jobs=N_JOBS if n_jobs is None else n_jobs)(
            delayed(_fit_predict)(pipe, X, y, tr, te, k, train_tf, test_tf)
            for k, (tr, te) in enumerate(splits))
        rows, oof = [], np.full((int(np.ceil(len(splits) / N_FOLDS)), len(y)), np.nan)
        for k, ((tr, te), p) in enumerate(zip(splits, probs)):
            r = fold_metrics(y[te], p); r['fold'] = k; rows.append(r)
            oof[k // N_FOLDS, te] = p
        RESULTS[name] = {'folds': pd.DataFrame(rows).set_index('fold'), 'oof': oof,
                         'seconds': time.time() - t0}
        pickle.dump(RESULTS[name], open(key, 'wb'))
    if verbose:
        f = RESULTS[name]['folds']
        print(f"  {name:<46} AUC={f.AUC.mean():.4f}  Acc={f.Accuracy.mean():.4f}  "
              f"Sens={f.Sensitivity.mean():.4f}  Spec={f.Specificity.mean():.4f}")
    return RESULTS[name]

# ── Statistics ─────────────────────────────────────────────────────────────
def corrected_t(diffs, ratio=TEST_TRAIN_RATIO):
    """Nadeau & Bengio (2003) corrected resampled t-test for repeated CV.
    Returns mean, 95% CI half-width, two-sided p-value."""
    d = np.asarray(diffs, float); d = d[~np.isnan(d)]; k = len(d)
    var = np.var(d, ddof=1) * (1 / k + ratio)
    se = np.sqrt(var) if var > 0 else 0.0
    t_crit = stats.t.ppf(0.975, k - 1)
    if se == 0:
        return d.mean(), 0.0, 1.0 if d.mean() == 0 else 0.0
    t = d.mean() / se
    return d.mean(), t_crit * se, 2 * stats.t.sf(abs(t), k - 1)

def holm(pvals):
    p = np.asarray(pvals, float); order = np.argsort(p); m = len(p)
    adj = np.empty(m); run = 0.0
    for i, idx in enumerate(order):
        run = max(run, (m - i) * p[idx]); adj[idx] = min(1.0, run)
    return adj

def calibration_slope(y, p):
    lp = np.log(np.clip(p, 1e-6, 1 - 1e-6) / (1 - np.clip(p, 1e-6, 1 - 1e-6)))
    lr = LogisticRegression(C=1e6, max_iter=1000).fit(lp[:, None], y)
    return float(lr.coef_[0, 0]), float(lr.intercept_[0])

def summarize(names, ref, metrics=('AUC', 'Accuracy', 'Sensitivity', 'Specificity', 'F1', 'Brier'),
              title=None, labels=None):
    """Table: mean [95% CI] per metric, ΔAUC vs `ref` with corrected CI and Holm p."""
    rows = []
    ref_f = RESULTS[ref]['folds']
    for n in names:
        f = RESULTS[n]['folds']; idx = f.index.intersection(ref_f.index)
        row = {'Variant': labels.get(n, n) if labels else n}
        for m in metrics:
            mu, hw, _ = corrected_t(f.loc[idx, m] - 0.0)
            row[m] = mu; row[f'{m}_lo'] = mu - hw; row[f'{m}_hi'] = mu + hw
        slopes = [calibration_slope(y_cl[~np.isnan(o)], o[~np.isnan(o)])[0]
                  for o in RESULTS[n]['oof'] if np.isfinite(o).sum() > 50]
        row['CalSlope'] = float(np.mean(slopes)) if slopes else np.nan
        for m in ('AUC', 'Accuracy', 'Sensitivity'):
            d, hw, p = corrected_t(f.loc[idx, m].values - ref_f.loc[idx, m].values)
            tag = 'Δ' + m
            row[tag] = d; row[f'{tag}_lo'] = d - hw; row[f'{tag}_hi'] = d + hw
            row[f'p_{m}'] = p if n != ref else np.nan
        rows.append(row)
    df = pd.DataFrame(rows).set_index('Variant')
    mask = df['p_AUC'].notna()
    if mask.any():
        df.loc[mask, 'p_AUC_holm'] = holm(df.loc[mask, 'p_AUC'])
        df.loc[mask, 'p_Accuracy_holm'] = holm(df.loc[mask, 'p_Accuracy'])
    if title:
        df.to_csv(f'{TAB_DIR}/{title}.csv')
    return df

def fmt_table(df, cols=('AUC', 'Accuracy', 'Sensitivity', 'Specificity', 'F1', 'Brier')):
    """Compact human-readable view: value [lo, hi]."""
    out = pd.DataFrame(index=df.index)
    for c in cols:
        out[c] = [f'{v:.3f} [{lo:.3f}, {hi:.3f}]' for v, lo, hi in
                  zip(df[c], df[f'{c}_lo'], df[f'{c}_hi'])]
    out['ΔAUC [95% CI]'] = [f'{v:+.3f} [{lo:+.3f}, {hi:+.3f}]' for v, lo, hi in
                            zip(df['ΔAUC'], df['ΔAUC_lo'], df['ΔAUC_hi'])]
    if 'p_AUC_holm' in df:
        out['p (Holm)'] = [('—' if np.isnan(p) else f'{p:.3f}') for p in df['p_AUC_holm']]
    if 'CalSlope' in df:
        out['Cal. slope'] = df['CalSlope'].round(2)
    return out

def forest_plot(df, title, fname, metric='ΔAUC', ref_label='reference'):
    fig, ax = plt.subplots(figsize=(8, 0.42 * len(df) + 1.4))
    yy = np.arange(len(df))[::-1]
    lo, hi, v = df[f'{metric}_lo'], df[f'{metric}_hi'], df[metric]
    sig = df.get('p_AUC_holm', pd.Series(np.nan, index=df.index)).fillna(1) < 0.05
    colors = ['#c0392b' if (s and val < 0) else '#1e8449' if (s and val > 0) else '#555'
              for s, val in zip(sig, v)]
    ax.hlines(yy, lo, hi, color=colors, lw=2)
    ax.scatter(v, yy, color=colors, zorder=3, s=28)
    ax.axvline(0, color='k', lw=0.8, ls='--')
    ax.set_yticks(yy); ax.set_yticklabels(df.index, fontsize=9)
    ax.set_xlabel(f'{metric} vs {ref_label} (95% corrected CI)')
    ax.set_title(title, fontweight='bold', fontsize=11)
    plt.tight_layout(); plt.savefig(f'{FIG_DIR}/{fname}', dpi=200, bbox_inches='tight')
    plt.show()

print(f"Protocol: {N_REPEATS}×{N_FOLDS}-fold repeated stratified CV = {len(SPLITS)} folds; "
      f"outputs → {ABL_DIR}")


### 17B-1. Component ablation
One component removed at a time; ΔAUC is paired by fold against the full system.

In [ ]:
# ══════════════════════════════════════════════════════════════════════════
# 17B-1. COMPONENT ABLATION (reviewer row 16)
# Remove one component at a time from the full system under the identical
# protocol. Full system = 13 clinical + 23 ECG-proxy features, SMOTE (training
# folds only), soft-voting over tuned XGB + SVM + GBM + RF.
# ══════════════════════════════════════════════════════════════════════════
print("── 17B-1 Component ablation ─────────────────────────────────────────")
ABL1 = {
    'Full system (clinical + ECG-proxy, SMOTE, soft voting)': make_pipe(FEAT_FULL()),
    # data / feature components
    '− ECG-proxy block (clinical only)':                     make_pipe(FEAT_CLINICAL()),
    '− clinical block (ECG-proxy only)':                     make_pipe(FEAT_FULL(keep_clinical=False)),
    '− duplicated raw proxy columns':                         make_pipe(FEAT_FULL(drop_dup_raw=True)),
    '− polynomial terms':        make_pipe(FEAT_FULL(groups=tuple(g for g in ALL_GROUPS if g != 'polynomial'))),
    '− interaction terms':       make_pipe(FEAT_FULL(groups=tuple(g for g in ALL_GROUPS if g != 'interaction'))),
    '− log transforms':          make_pipe(FEAT_FULL(groups=tuple(g for g in ALL_GROUPS if g != 'log'))),
    '− ratio features':          make_pipe(FEAT_FULL(groups=tuple(g for g in ALL_GROUPS if g != 'ratio'))),
    '− threshold flags':         make_pipe(FEAT_FULL(groups=tuple(g for g in ALL_GROUPS if g != 'threshold'))),
    '− HR-reserve features':     make_pipe(FEAT_FULL(groups=tuple(g for g in ALL_GROUPS if g != 'hr_reserve'))),
    '− ischaemia composite':     make_pipe(FEAT_FULL(groups=tuple(g for g in ALL_GROUPS if g != 'composite'))),
    'downslope coding fixed (slope==3)':                       make_pipe(FEAT_FULL(fix_downslope=True)),
    # imbalance handling
    '− SMOTE (no resampling)':                                 make_pipe(FEAT_FULL(), sampler=None),
    # ensemble / classifier
    '− soft voting → XGBoost only':                            make_pipe(FEAT_FULL(), clf='xgb'),
    '− soft voting → SVM only':                                make_pipe(FEAT_FULL(), clf='svm'),
    '− soft voting → GBM only':                                make_pipe(FEAT_FULL(), clf='gbm'),
    '− soft voting → RF only':                                 make_pipe(FEAT_FULL(), clf='rf'),
    '− soft voting → logistic regression':                     make_pipe(FEAT_FULL(), clf='lr'),
    'soft → hard (majority) voting':                           make_pipe(FEAT_FULL(), clf='hard_voting'),
    'soft voting → stacking (LR meta-learner)':                make_pipe(FEAT_FULL(), clf='stacking'),
    # minimal systems
    'Clinical only, no SMOTE, logistic regression':            make_pipe(FEAT_CLINICAL(), sampler=None, clf='lr'),
}
for name, pipe in ABL1.items():
    evaluate(name, pipe)

REF = 'Full system (clinical + ECG-proxy, SMOTE, soft voting)'
abl1 = summarize(list(ABL1), REF, title='E1_component_ablation')
display(fmt_table(abl1))
forest_plot(abl1.drop(REF), 'Figure A1: Component ablation — ΔAUC vs full system',
            'figA1_component_ablation.png', ref_label='full system')


### 17B-2. Control experiments
Does the ECG-proxy block beat equally sized blocks of non-ECG features, pure noise, or a row-shuffled copy of itself?

In [ ]:
# ══════════════════════════════════════════════════════════════════════════
# 17B-2. CONTROL EXPERIMENTS — is the ECG-proxy block special?
# If the proxy features carry genuine extra signal, the full system should beat
#   (a) clinical + 18 engineered NON-ECG features (same transform types),
#   (b) clinical + 18 pure-noise columns,
#   (c) clinical + the proxy block row-shuffled (realistic marginals, no link
#       to the patient).
# A label-permutation test checks that the full system is above chance at all.
# ══════════════════════════════════════════════════════════════════════════
print("── 17B-2 Control experiments ────────────────────────────────────────")
ABL2 = {
    REF: ABL1[REF],
    'Clinical only': ABL1['− ECG-proxy block (clinical only)'],
    'Clinical + 18 engineered non-ECG features': make_pipe(FEAT_CLINICAL(extra='nonecg')),
    'Clinical + 18 Gaussian-noise features':     make_pipe(FEAT_CLINICAL(extra='noise')),
    'Clinical + row-shuffled ECG-proxy block':   make_pipe(FEAT_FULL(drop_dup_raw=True, extra='shuffled_proxy')),
    'Clinical + ECG-proxy + non-ECG features':   make_pipe(FEAT_FULL(extra='nonecg')),
}
for name, pipe in ABL2.items():
    evaluate(name, pipe)
abl2 = summarize(list(ABL2), REF, title='E2_controls')
display(fmt_table(abl2))

# ── Label-permutation test (Ojala & Garriga 2010) ─────────────────────────
N_PERM = int(os.environ.get('MCF_NPERM', 100))
perm_key = f'{ABL_DIR}/cache/perm_{N_PERM}.pkl'
skf5 = list(StratifiedKFold(5, shuffle=True, random_state=SEED).split(X_cl_raw, y_cl))

def _perm_auc(pipe, yy):
    p = np.zeros(len(yy))
    for tr, te in skf5:
        p[te] = clone(pipe).fit(X_cl_raw[tr], yy[tr]).predict_proba(X_cl_raw[te])[:, 1]
    return roc_auc_score(yy, p)

if os.path.exists(perm_key):
    obs_auc, null_auc = pickle.load(open(perm_key, 'rb'))
else:
    rng = np.random.RandomState(SEED)
    obs_auc = _perm_auc(ABL1[REF], y_cl)
    null_auc = np.array(Parallel(n_jobs=N_JOBS)(
        delayed(_perm_auc)(ABL1[REF], rng.permutation(y_cl)) for _ in range(N_PERM)))
    pickle.dump((obs_auc, null_auc), open(perm_key, 'wb'))
p_perm = (1 + np.sum(null_auc >= obs_auc)) / (1 + len(null_auc))
print(f"\nPermutation test: observed AUC={obs_auc:.3f}; null AUC={null_auc.mean():.3f}±{null_auc.std():.3f}; "
      f"p={p_perm:.4f} ({len(null_auc)} permutations)")
pd.DataFrame({'observed_auc': [obs_auc], 'null_mean': [null_auc.mean()], 'null_sd': [null_auc.std()],
              'null_max': [null_auc.max()], 'p_value': [p_perm], 'n_perm': [len(null_auc)]}
             ).to_csv(f'{TAB_DIR}/E2_permutation_test.csv', index=False)

fig, axes = plt.subplots(1, 2, figsize=(13, 4.2), gridspec_kw={'width_ratios': [1.4, 1]})
yy = np.arange(len(abl2))[::-1]
axes[0].errorbar(abl2['AUC'], yy, xerr=[abl2['AUC'] - abl2['AUC_lo'], abl2['AUC_hi'] - abl2['AUC']],
                 fmt='o', color='#2c3e50', capsize=3)
axes[0].set_yticks(yy); axes[0].set_yticklabels(abl2.index, fontsize=9)
axes[0].set_xlabel('AUC (95% corrected CI)'); axes[0].set_title('Feature-block controls', fontweight='bold')
axes[1].hist(null_auc, bins=20, color='#bdc3c7', edgecolor='k')
axes[1].axvline(obs_auc, color='#c0392b', lw=2, label=f'observed {obs_auc:.3f}')
axes[1].set_xlabel('AUC under permuted labels'); axes[1].legend()
axes[1].set_title(f'Label-permutation test (p={p_perm:.3f})', fontweight='bold')
plt.suptitle('Figure A2: Control experiments', fontweight='bold')
plt.tight_layout(); plt.savefig(f'{FIG_DIR}/figA2_controls.png', dpi=200, bbox_inches='tight'); plt.show()


### 17B-3. Class-imbalance strategies
Resampling is done inside training folds only. Also checks how realistic SMOTE patients are.

In [ ]:
# ══════════════════════════════════════════════════════════════════════════
# 17B-3. CLASS-IMBALANCE STRATEGIES (reviewer rows 12 & 15)
# SMOTE on mixed continuous/binary/ordinal data can create unrealistic
# patients. Compare: no correction, class weighting, random oversampling,
# SMOTE (after features, as in the paper), SMOTE on raw rows before feature
# derivation, SMOTE-NC (categorical-aware), Borderline-SMOTE and ADASYN.
# All resampling happens inside training folds only.
# ══════════════════════════════════════════════════════════════════════════
print("── 17B-3 Resampling strategies ──────────────────────────────────────")
ABL3 = {
    REF: ABL1[REF],
    'No correction':                         make_pipe(FEAT_FULL(), sampler=None),
    'Class weighting (no resampling)':       make_pipe(FEAT_FULL(), sampler=None, balanced=True),
    'Random oversampling':                   make_pipe(FEAT_FULL(), sampler='ros'),
    'SMOTE on raw rows → derive proxy':      make_pipe(FEAT_FULL(), sampler='smote', sampler_stage='pre'),
    'SMOTE-NC on raw rows → derive proxy':   make_pipe(FEAT_FULL(), sampler='smotenc', sampler_stage='pre'),
    'Borderline-SMOTE':                      make_pipe(FEAT_FULL(), sampler='borderline'),
    'ADASYN':                                make_pipe(FEAT_FULL(), sampler='adasyn'),
}
for name, pipe in ABL3.items():
    evaluate(name, pipe)
abl3 = summarize(list(ABL3), REF, title='E3_resampling')
display(fmt_table(abl3))

# ── How realistic are SMOTE patients? Share of synthetic rows with impossible
#    (non-integer) categorical codes, and derived-feature inconsistency ──────
tr, te = SPLITS[0]
imp = SimpleImputer(strategy='median').fit(X_cl_raw[tr])
Xtr_imp = imp.transform(X_cl_raw[tr])
pf = ProxyFeatures().fit(None)
Z = pf.transform(Xtr_imp); sc = StandardScaler().fit(Z)
Zs, ys = SMOTE(random_state=SEED).fit_resample(sc.transform(Z), y_cl[tr])
synth = sc.inverse_transform(Zs[len(tr):])
names = pf.feature_names_
cat_idx = [names.index(c) for c in CAT_COLS]
non_integer = np.mean(np.any(np.abs(synth[:, cat_idx] - np.round(synth[:, cat_idx])) > 1e-6, axis=1))
op, op_sq = synth[:, names.index('oldpeak')], synth[:, names.index('oldpeak_sq')]
inconsistent = np.mean(np.abs(op ** 2 - op_sq) > 0.05 * np.maximum(1, op_sq))
dup_gap = np.mean(np.abs(synth[:, names.index('oldpeak')] - synth[:, names.index('proxy:oldpeak')]) > 1e-6)
realism = pd.DataFrame({
    'Synthetic rows (fold 1)': [len(synth)],
    'Rows with non-integer categorical codes': [f'{non_integer:.1%}'],
    'Rows where oldpeak_sq ≠ oldpeak² (>5%)': [f'{inconsistent:.1%}'],
    'Rows where duplicate oldpeak copies disagree': [f'{dup_gap:.1%}'],
})
display(realism); realism.to_csv(f'{TAB_DIR}/E3_smote_realism.csv', index=False)

# ── Class counts per partition (reviewer row 15) ─────────────────────────
X_tr_h, X_te_h, y_tr_h, y_te_h = train_test_split(X_cl_raw, y_cl, test_size=0.2,
                                                  stratify=y_cl, random_state=SEED)
_, y_after = SMOTE(random_state=SEED).fit_resample(
    np.nan_to_num(SimpleImputer(strategy='median').fit_transform(X_tr_h)), y_tr_h)
counts = pd.DataFrame({
    'Dataset': ['D1/D3 Cleveland (80/20 hold-out)', 'D1/D3 Cleveland (one 10-fold CV fold)'],
    'Train (healthy/disease)': [f'{np.sum(y_tr_h == 0)}/{np.sum(y_tr_h == 1)}',
                                f'{np.sum(y_cl[tr] == 0)}/{np.sum(y_cl[tr] == 1)}'],
    'Train after SMOTE': [f'{np.sum(y_after == 0)}/{np.sum(y_after == 1)}',
                          f'{np.sum(ys == 0)}/{np.sum(ys == 1)}'],
    'Test (healthy/disease)': [f'{np.sum(y_te_h == 0)}/{np.sum(y_te_h == 1)}',
                               f'{np.sum(y_cl[te] == 0)}/{np.sum(y_cl[te] == 1)}'],
})
display(counts); counts.to_csv(f'{TAB_DIR}/E3_class_counts.csv', index=False)


### 17B-4. Evaluation protocol and leakage
How much do leaky preprocessing, SMOTE-before-CV, tuning choices and the choice of hold-out split change the numbers?

In [ ]:
# ══════════════════════════════════════════════════════════════════════════
# 17B-4. EVALUATION PROTOCOL & LEAKAGE (reviewer rows 8, 13, 16)
#  (a) preprocessing fitted on all 303 rows before CV (as in Sections 4–9)
#  (b) SMOTE applied to the whole dataset before CV (classic leakage)
#  (c) nested hyper-parameter tuning vs the fixed Section-15 values
#  (d) hold-out split sensitivity: 200 random 80/20 splits instead of seed 42
# ══════════════════════════════════════════════════════════════════════════
print("── 17B-4 Protocol and leakage ───────────────────────────────────────")
from sklearn.model_selection import RandomizedSearchCV

def ci_row(name, label=None):
    f = RESULTS[name]['folds']; row = {'Variant': label or name}
    for m in ('AUC', 'Accuracy', 'Sensitivity', 'Specificity'):
        mu, hw, _ = corrected_t(f[m]); row[m] = f'{mu:.3f} [{mu - hw:.3f}, {mu + hw:.3f}]'
    return row

# (a) global preprocessing (imputation + proxy + scaling fitted on all rows)
X_glob = StandardScaler().fit_transform(
    ProxyFeatures().fit(None).transform(SimpleImputer(strategy='median').fit_transform(X_cl_raw)))
leaky_a = ImbPipeline([('sampler', SMOTE(random_state=SEED)), ('clf', make_clf('voting'))])
evaluate('Leaky (a): preprocessing fitted on all rows', leaky_a, X=X_glob)

# (b) SMOTE on the full dataset, then CV (synthetic neighbours of test rows in training)
X_sm_all, y_sm_all = SMOTE(random_state=SEED).fit_resample(X_glob, y_cl)
splits_sm = list(RepeatedStratifiedKFold(n_splits=N_FOLDS, n_repeats=N_REPEATS,
                                         random_state=SEED).split(X_sm_all, y_sm_all))
evaluate('Leaky (b): SMOTE before CV', ImbPipeline([('clf', make_clf('voting'))]),
         X=X_sm_all, y=y_sm_all, splits=splits_sm)

# (c) nested tuning of XGBoost (inner 3-fold randomized search, 20 candidates)
xgb_space = {'clf__n_estimators': [200, 300, 500], 'clf__max_depth': [2, 3, 4],
             'clf__learning_rate': [0.01, 0.05, 0.1], 'clf__subsample': [0.7, 0.8, 1.0],
             'clf__colsample_bytree': [0.7, 0.8, 1.0], 'clf__min_child_weight': [1, 3, 5],
             'clf__gamma': [0, 0.1, 0.2]}
nested = RandomizedSearchCV(make_pipe(FEAT_FULL(), clf='xgb'), xgb_space, n_iter=20, cv=3,
                            scoring='roc_auc', random_state=SEED, n_jobs=1)
evaluate('XGBoost, nested tuning (inner 3-fold)', nested)
evaluate('XGBoost, library defaults', make_pipe(FEAT_FULL(), clf=XGBClassifier(
    eval_metric='logloss', random_state=SEED, verbosity=0, n_jobs=1)))
evaluate('− soft voting → XGBoost only', ABL1['− soft voting → XGBoost only'], verbose=False)

prot = pd.DataFrame([ci_row(REF, 'Leak-free CV (reference)'),
                     ci_row('Leaky (a): preprocessing fitted on all rows'),
                     ci_row('Leaky (b): SMOTE before CV'),
                     ci_row('− soft voting → XGBoost only', 'XGBoost, fixed Section-15 params'),
                     ci_row('XGBoost, nested tuning (inner 3-fold)'),
                     ci_row('XGBoost, library defaults')]).set_index('Variant')
display(prot); prot.to_csv(f'{TAB_DIR}/E4_protocol_leakage.csv')

# (d) hold-out sensitivity: the paper's 0.8852 comes from ONE 80/20 split
N_SPLITS_HO = int(os.environ.get('MCF_HOLDOUT_SPLITS', 200))
ho_key = f'{ABL_DIR}/cache/holdout_{N_SPLITS_HO}.pkl'

def _holdout(seed):
    tr, te = train_test_split(np.arange(len(y_cl)), test_size=0.2, stratify=y_cl, random_state=seed)
    out = {'seed': seed}
    for tag, pipe in (('fused', ABL1[REF]), ('clinical', ABL1['− ECG-proxy block (clinical only)'])):
        p = clone(pipe).fit(X_cl_raw[tr], y_cl[tr]).predict_proba(X_cl_raw[te])[:, 1]
        mm = fold_metrics(y_cl[te], p)
        out[f'{tag}_AUC'], out[f'{tag}_Acc'] = mm['AUC'], mm['Accuracy']
    return out

if os.path.exists(ho_key):
    ho = pickle.load(open(ho_key, 'rb'))
else:
    ho = pd.DataFrame(Parallel(n_jobs=N_JOBS)(delayed(_holdout)(s) for s in range(N_SPLITS_HO)))
    pickle.dump(ho, open(ho_key, 'wb'))
ho['dAcc'] = ho.fused_Acc - ho.clinical_Acc; ho['dAUC'] = ho.fused_AUC - ho.clinical_AUC
s42 = ho.loc[ho.seed == 42].iloc[0]
q = lambda s: f'{s.mean():.3f} (2.5–97.5%: {s.quantile(.025):.3f}–{s.quantile(.975):.3f})'
ho_summary = pd.DataFrame({
    'Metric': ['Fused accuracy', 'Clinical accuracy', 'Fused − clinical accuracy',
               'Fused AUC', 'Clinical AUC', 'Fused − clinical AUC'],
    f'Over {N_SPLITS_HO} random 80/20 splits': [q(ho.fused_Acc), q(ho.clinical_Acc), q(ho.dAcc),
                                                  q(ho.fused_AUC), q(ho.clinical_AUC), q(ho.dAUC)],
    'Seed 42 (paper split)': [f'{s42.fused_Acc:.3f}', f'{s42.clinical_Acc:.3f}', f'{s42.dAcc:+.3f}',
                              f'{s42.fused_AUC:.3f}', f'{s42.clinical_AUC:.3f}', f'{s42.dAUC:+.3f}'],
    'Percentile of seed 42': [f'{stats.percentileofscore(ho[c], s42[c]):.0f}'
                              for c in ('fused_Acc', 'clinical_Acc', 'dAcc', 'fused_AUC', 'clinical_AUC', 'dAUC')],
}).set_index('Metric')
display(ho_summary); ho_summary.to_csv(f'{TAB_DIR}/E4_holdout_sensitivity.csv')
print(f"Fused beats clinical (accuracy) on {np.mean(ho.dAcc > 0):.0%} of splits, ties on "
      f"{np.mean(ho.dAcc == 0):.0%}, loses on {np.mean(ho.dAcc < 0):.0%}.")

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for ax, (a, b, lab) in zip(axes, [('fused_Acc', 'clinical_Acc', 'Accuracy'), ('fused_AUC', 'clinical_AUC', 'AUC')]):
    ax.hist(ho[a], bins=25, alpha=.6, label='Fused (full system)', color='#2980b9')
    ax.hist(ho[b], bins=25, alpha=.6, label='Clinical only', color='#e67e22')
    ax.axvline(s42[a], color='#2980b9', ls='--', lw=2); ax.axvline(s42[b], color='#e67e22', ls='--', lw=2)
    ax.set_xlabel(f'Hold-out {lab} (61 patients)'); ax.legend(fontsize=8)
    ax.set_title(f'{lab}: {N_SPLITS_HO} random splits (dashed = seed 42)', fontweight='bold', fontsize=10)
plt.suptitle('Figure A3: Hold-out estimates depend strongly on the split', fontweight='bold')
plt.tight_layout(); plt.savefig(f'{FIG_DIR}/figA3_holdout_sensitivity.png', dpi=200, bbox_inches='tight'); plt.show()


### 17B-5. Robustness and edge cases
Label noise, missing values, measurement noise, smaller training sets, unavailable tests, and patient subgroups.

In [ ]:
# ══════════════════════════════════════════════════════════════════════════
# 17B-5. ROBUSTNESS & EDGE CASES — does the ECG-proxy block help when the
# data are harder?  Each scenario is applied identically (same random seed per
# fold) to the full system and to the clinical-only system, so the comparison
# is paired.  3 × 10-fold CV to bound the run time.
# ══════════════════════════════════════════════════════════════════════════
print("── 17B-5 Robustness and edge cases ──────────────────────────────────")
ROB_SPLITS = SPLITS[:3 * N_FOLDS]
FUSED_PIPE, CLIN_PIPE = ABL1[REF], ABL1['− ECG-proxy block (clinical only)']
col = {c: i for i, c in enumerate(CLIN_COLS)}

def flip_labels(rate):
    def tf(X, y, seed):
        rng = np.random.RandomState(seed); y = y.copy()
        idx = rng.choice(len(y), int(round(rate * len(y))), replace=False); y[idx] = 1 - y[idx]
        return X, y
    return tf

def mcar(rate):
    def tf(X, seed):
        rng = np.random.RandomState(seed); X = X.copy(); X[rng.rand(*X.shape) < rate] = np.nan
        return X
    return tf

def mcar_train(rate):
    f = mcar(rate)
    return lambda X, y, seed: (f(X, seed), y)

def gauss_noise(sigma):
    sd = np.nanstd(X_cl_raw[:, [col[c] for c in CONT_COLS]], axis=0)
    def tf(X, seed):
        rng = np.random.RandomState(seed); X = X.copy()
        for j, c in enumerate(CONT_COLS):
            X[:, col[c]] += rng.randn(len(X)) * sigma * sd[j]
        return X
    return tf

def subsample(frac):
    def tf(X, y, seed):
        idx, _ = train_test_split(np.arange(len(y)), train_size=frac, stratify=y, random_state=seed)
        return X[idx], y[idx]
    return tf

def unavailable(cols):          # feature never measured (train AND test)
    def tf(X, seed=None):
        X = X.copy(); X[:, [col[c] for c in cols]] = np.nan; return X
    return tf

def unavailable_train(cols):
    f = unavailable(cols)
    return lambda X, y, seed: (f(X), y)

NO_ANGIO, NO_EXERCISE = ['ca', 'thal'], ['thalach', 'exang', 'oldpeak', 'slope']
SCENARIOS = {'Baseline (no perturbation)': (None, None)}
for r in (0.05, 0.10, 0.20):
    SCENARIOS[f'Training label noise {int(r * 100)}%'] = (flip_labels(r), None)
for r in (0.10, 0.20, 0.30):
    SCENARIOS[f'Test-time missing values (MCAR) {int(r * 100)}%'] = (None, mcar(r))
SCENARIOS['Missing values in train AND test 20%'] = (mcar_train(0.2), mcar(0.2))
for s in (0.25, 0.5, 1.0):
    SCENARIOS[f'Measurement noise σ={s}×SD (test)'] = (None, gauss_noise(s))
for fr in (0.25, 0.50, 0.75):
    SCENARIOS[f'Training set reduced to {int(fr * 100)}%'] = (subsample(fr), None)
SCENARIOS['No angiography/thallium (ca, thal unavailable)'] = (unavailable_train(NO_ANGIO), unavailable(NO_ANGIO))
SCENARIOS['No exercise test (thalach, exang, oldpeak, slope)'] = (unavailable_train(NO_EXERCISE), unavailable(NO_EXERCISE))
SCENARIOS['Resting data only (both unavailable)'] = (unavailable_train(NO_ANGIO + NO_EXERCISE),
                                                     unavailable(NO_ANGIO + NO_EXERCISE))
rob_rows = []
for sc_name, (trf, tef) in SCENARIOS.items():
    for tag, pipe in (('fused', FUSED_PIPE), ('clinical', CLIN_PIPE)):
        evaluate(f'[rob] {sc_name} | {tag}', pipe, splits=ROB_SPLITS, train_tf=trf, test_tf=tef,
                 verbose=False)
    fu = RESULTS[f'[rob] {sc_name} | fused']['folds']; cl = RESULTS[f'[rob] {sc_name} | clinical']['folds']
    d, hw, p = corrected_t(fu.AUC.values - cl.AUC.values)
    da, hwa, pa = corrected_t(fu.Accuracy.values - cl.Accuracy.values)
    rob_rows.append({'Scenario': sc_name, 'AUC fused': fu.AUC.mean(), 'AUC clinical': cl.AUC.mean(),
                     'ΔAUC': d, 'ΔAUC_lo': d - hw, 'ΔAUC_hi': d + hw, 'p_AUC': p,
                     'Sens fused': fu.Sensitivity.mean(), 'Sens clinical': cl.Sensitivity.mean(),
                     'ΔAcc': da, 'p_Acc': pa})
rob = pd.DataFrame(rob_rows).set_index('Scenario')
rob['p_AUC_holm'] = holm(rob['p_AUC'].values)
rob.to_csv(f'{TAB_DIR}/E5_robustness.csv')
show = rob[['AUC fused', 'AUC clinical']].round(3)
show['ΔAUC [95% CI]'] = [f'{v:+.3f} [{lo:+.3f}, {hi:+.3f}]' for v, lo, hi in zip(rob['ΔAUC'], rob['ΔAUC_lo'], rob['ΔAUC_hi'])]
show['p (Holm)'] = rob['p_AUC_holm'].round(3)
show['Sens fused / clinical'] = [f'{a:.3f} / {b:.3f}' for a, b in zip(rob['Sens fused'], rob['Sens clinical'])]
display(show)
forest_plot(rob,'Figure A4: Fused − clinical ΔAUC under perturbations',
            'figA4_robustness.png', ref_label='clinical only (same perturbation)')

# ── Subgroup analysis on out-of-fold predictions (averaged over repeats) ─────
p_fu = np.nanmean(RESULTS[REF]['oof'], axis=0)
p_cl = np.nanmean(RESULTS['− ECG-proxy block (clinical only)']['oof'], axis=0)
GROUPS = {'All patients': np.ones(len(y_cl), bool),
          'Female': Xdf_cl.sex.values == 0, 'Male': Xdf_cl.sex.values == 1,
          'Age < 55': Xdf_cl.age.values < 55, 'Age ≥ 55': Xdf_cl.age.values >= 55,
          'Asymptomatic chest pain (cp=4)': Xdf_cl.cp.values == 4,
          'Symptomatic chest pain (cp=1–3)': Xdf_cl.cp.values != 4,
          'Normal resting ECG': Xdf_cl.restecg.values == 0,
          'Abnormal resting ECG': Xdf_cl.restecg.values > 0}

def boot_auc(mask, B=2000, seed=SEED):
    rng = np.random.RandomState(seed); idx = np.where(mask)[0]; yy = y_cl[idx]
    a_f, a_c, d = [], [], []
    for _ in range(B):
        b = rng.choice(idx, len(idx), replace=True)
        if len(np.unique(y_cl[b])) < 2: continue
        af, ac = roc_auc_score(y_cl[b], p_fu[b]), roc_auc_score(y_cl[b], p_cl[b])
        a_f.append(af); a_c.append(ac); d.append(af - ac)
    lo, hi = np.percentile(d, [2.5, 97.5])
    return {'n': len(idx), 'disease': int(yy.sum()), 'AUC fused': roc_auc_score(yy, p_fu[idx]),
            'AUC clinical': roc_auc_score(yy, p_cl[idx]),
            'ΔAUC': roc_auc_score(yy, p_fu[idx]) - roc_auc_score(yy, p_cl[idx]),
            'ΔAUC 95% CI': f'[{lo:+.3f}, {hi:+.3f}]'}

sub = pd.DataFrame({g: boot_auc(m) for g, m in GROUPS.items()}).T
display(sub); sub.to_csv(f'{TAB_DIR}/E5_subgroups.csv')


### 17B-6. Ordinal encoding and probability calibration

In [ ]:
# ══════════════════════════════════════════════════════════════════════════
# 17B-6. ORDINAL ENCODING & PROBABILITY CALIBRATION (reviewer rows 8d, 13)
#  • cp, restecg, slope, thal are nominal/ordinal codes but are standardised as
#    if continuous; compare with one-hot encoding.
#  • SMOTE trains on a 50/50 class mix while the real prevalence is 45.9%, which
#    can distort probabilities. Compare Brier score and calibration slope
#    (ideal = 1) with and without SMOTE and with Platt / isotonic calibration.
# ══════════════════════════════════════════════════════════════════════════
print("── 17B-6 Encoding and calibration ───────────────────────────────────")
from sklearn.calibration import CalibratedClassifierCV, calibration_curve

ABL6 = {
    REF: ABL1[REF],
    'One-hot cp/restecg/slope/thal (full system)':  make_pipe(FEAT_FULL(onehot=True)),
    'One-hot cp/restecg/slope/thal (clinical only)': make_pipe(FEAT_CLINICAL(onehot=True)),
    'Clinical only': ABL1['− ECG-proxy block (clinical only)'],
    'Platt-calibrated soft voting':    make_pipe(FEAT_FULL(), clf=CalibratedClassifierCV(
                                                 make_clf('voting'), method='sigmoid', cv=5)),
    'Isotonic-calibrated soft voting': make_pipe(FEAT_FULL(), clf=CalibratedClassifierCV(
                                                 make_clf('voting'), method='isotonic', cv=5)),
    '− SMOTE (no resampling)': ABL1['− SMOTE (no resampling)'],
}
for name, pipe in ABL6.items():
    evaluate(name, pipe)
abl6 = summarize(list(ABL6), REF, title='E6_encoding_calibration')
display(fmt_table(abl6))

fig, ax = plt.subplots(figsize=(6.2, 5.5))
ax.plot([0, 1], [0, 1], 'k--', lw=1, label='perfect calibration')
for name, c in [(REF, '#2980b9'), ('− SMOTE (no resampling)', '#27ae60'),
                ('Isotonic-calibrated soft voting', '#8e44ad'), ('Clinical only', '#e67e22')]:
    o = RESULTS[name]['oof'][0]
    fr, mp = calibration_curve(y_cl, o, n_bins=8, strategy='quantile')
    ax.plot(mp, fr, 'o-', color=c, label=f"{name[:38]} (Brier {brier_score_loss(y_cl, o):.3f})")
ax.set_xlabel('Mean predicted probability'); ax.set_ylabel('Observed disease fraction')
ax.set_title('Figure A5: Reliability (out-of-fold, repeat 1)', fontweight='bold'); ax.legend(fontsize=7.5)
plt.tight_layout(); plt.savefig(f'{FIG_DIR}/figA5_calibration.png', dpi=200, bbox_inches='tight'); plt.show()


### 17B-7. External validation (Hungary, Switzerland, VA Long Beach)
Trained on Cleveland only; the other sites have different prevalence and many missing tests.

In [ ]:
# ══════════════════════════════════════════════════════════════════════════
# 17B-7. EXTERNAL VALIDATION ON THE OTHER UCI SITES (distribution shift)
# Train on all 303 Cleveland patients; test unchanged on Hungary (Budapest),
# Switzerland (Zurich/Basel) and VA Long Beach. These sites have different
# disease prevalence and many missing tests (ca, thal, slope), so they are a
# realistic stress test. Missing values are imputed with Cleveland medians.
# ══════════════════════════════════════════════════════════════════════════
print("── 17B-7 External validation ────────────────────────────────────────")
SITES = {s: load_uci_site(s) for s in ('hungarian', 'switzerland', 'va')}
SITE_LABEL = {'hungarian': 'Hungary', 'switzerland': 'Switzerland', 'va': 'VA Long Beach'}

miss = pd.DataFrame({SITE_LABEL.get(s, s): {
    'n': len(y), 'disease prevalence': f'{y.mean():.1%}',
    **{f'missing {c}': f'{np.isnan(X[c]).mean():.0%}' for c in ('ca', 'thal', 'slope', 'chol', 'fbs', 'oldpeak')}}
    for s, (X, y) in [('Cleveland', (Xdf_cl, y_cl))] + list(SITES.items())}).T
display(miss); miss.to_csv(f'{TAB_DIR}/E7_site_characteristics.csv')

REDUCED = ('ca', 'thal', 'slope')         # mostly unavailable outside Cleveland
def drop_cols(cols):
    return lambda X: np.where(np.isin(np.arange(X.shape[1]), [CLIN_COLS.index(c) for c in cols]), np.nan, X)

EXT_MODELS = {
    'Fused (full system)': (ABL1[REF], None),
    'Clinical only': (ABL1['− ECG-proxy block (clinical only)'], None),
    'Fused, without ca/thal/slope': (ABL1[REF], drop_cols(REDUCED)),
    'Clinical, without ca/thal/slope': (ABL1['− ECG-proxy block (clinical only)'], drop_cols(REDUCED)),
}

def boot_ci(y, p, B=2000, seed=SEED):
    rng = np.random.RandomState(seed); vals = []
    for _ in range(B):
        b = rng.randint(0, len(y), len(y))
        if len(np.unique(y[b])) == 2: vals.append(roc_auc_score(y[b], p[b]))
    return np.percentile(vals, [2.5, 97.5])

def boot_delta(y, p1, p2, B=2000, seed=SEED):
    rng = np.random.RandomState(seed); vals = []
    for _ in range(B):
        b = rng.randint(0, len(y), len(y))
        if len(np.unique(y[b])) == 2: vals.append(roc_auc_score(y[b], p1[b]) - roc_auc_score(y[b], p2[b]))
    vals = np.array(vals); lo, hi = np.percentile(vals, [2.5, 97.5])
    return lo, hi, 2 * min(np.mean(vals <= 0), np.mean(vals >= 0))

ext_rows, ext_probs = [], {}
for mname, (pipe, tf) in EXT_MODELS.items():
    Xtr = X_cl_raw if tf is None else tf(X_cl_raw)
    model = clone(pipe).fit(Xtr, y_cl)
    for site, (Xs, ys) in SITES.items():
        Xte = Xs.values if tf is None else tf(Xs.values)
        p = model.predict_proba(Xte)[:, 1]; ext_probs[(mname, site)] = p
        mm = fold_metrics(ys, p); lo, hi = boot_ci(ys, p)
        ext_rows.append({'Model': mname, 'Site': SITE_LABEL[site], 'n': len(ys), 'AUC': mm['AUC'],
                         'AUC 95% CI': f'[{lo:.3f}, {hi:.3f}]', 'Accuracy': mm['Accuracy'],
                         'Sensitivity': mm['Sensitivity'], 'Specificity': mm['Specificity'],
                         'Brier': mm['Brier']})
ext = pd.DataFrame(ext_rows).set_index(['Site', 'Model']).sort_index()
display(ext.round(3)); ext.to_csv(f'{TAB_DIR}/E7_external_validation.csv')

delta_rows = []
for site, (Xs, ys) in SITES.items():
    for a, b in (('Fused (full system)', 'Clinical only'),
                 ('Fused, without ca/thal/slope', 'Clinical, without ca/thal/slope')):
        lo, hi, p = boot_delta(ys, ext_probs[(a, site)], ext_probs[(b, site)])
        d = roc_auc_score(ys, ext_probs[(a, site)]) - roc_auc_score(ys, ext_probs[(b, site)])
        delta_rows.append({'Site': SITE_LABEL[site], 'Comparison': f'{a} − {b}', 'ΔAUC': d,
                           '95% CI': f'[{lo:+.3f}, {hi:+.3f}]', 'bootstrap p': p})
ext_delta = pd.DataFrame(delta_rows).set_index(['Site', 'Comparison'])
display(ext_delta.round(3)); ext_delta.to_csv(f'{TAB_DIR}/E7_external_delta.csv')

# Edge case: cholesterol recorded as 0 (Switzerland/VA) — missing vs literal value
chol_rows = []
for site in ('switzerland', 'va'):
    Xs0, ys0 = load_uci_site(site, chol_zero_as_missing=False)
    for mname in ('Fused (full system)', 'Clinical only'):
        m = clone(EXT_MODELS[mname][0]).fit(X_cl_raw, y_cl)
        chol_rows.append({'Site': SITE_LABEL[site], 'Model': mname,
                          'AUC, chol=0 treated as missing': roc_auc_score(SITES[site][1], ext_probs[(mname, site)]),
                          'AUC, chol=0 taken literally': roc_auc_score(ys0, m.predict_proba(Xs0.values)[:, 1]),
                          'rows with chol=0': int((Xs0.chol == 0).sum())})
chol_df = pd.DataFrame(chol_rows).set_index(['Site', 'Model'])
display(chol_df.round(3)); chol_df.to_csv(f'{TAB_DIR}/E7_chol_zero_edge_case.csv')

fig, ax = plt.subplots(figsize=(9, 4))
w = 0.2
for i, mname in enumerate(EXT_MODELS):
    vals = [ext.loc[(SITE_LABEL[s], mname), 'AUC'] for s in SITES]
    ax.bar(np.arange(3) + (i - 1.5) * w, vals, w, label=mname)
ax.axhline(RESULTS[REF]['folds'].AUC.mean(), color='k', ls='--', lw=1, label='Cleveland internal CV (fused)')
ax.set_xticks(range(3)); ax.set_xticklabels([SITE_LABEL[s] for s in SITES]); ax.set_ylim(0.4, 1.0)
ax.set_ylabel('External AUC'); ax.legend(fontsize=8, ncol=2)
ax.set_title('Figure A6: External validation (trained on Cleveland only)', fontweight='bold')
plt.tight_layout(); plt.savefig(f'{FIG_DIR}/figA6_external.png', dpi=200, bbox_inches='tight'); plt.show()


### 17B-8. SHAP stability and agreement with permutation importance

In [ ]:
# ══════════════════════════════════════════════════════════════════════════
# 17B-8. SHAP STABILITY & AGREEMENT (reviewer rows 6 & 18)
# SHAP is post-hoc and associational; with N=303 and correlated / duplicated
# proxy features its rankings may be unstable. We measure:
#   • fold-to-fold Spearman correlation of global mean|SHAP| rankings
#   • seed-to-seed stability (10 XGBoost seeds, same data)
#   • agreement with permutation importance (AUC drop) on the same test folds
#   • how attribution is split between duplicated columns
# SHAP is computed for the XGBoost member (TreeSHAP), as in Section 18.
# ══════════════════════════════════════════════════════════════════════════
print("── 17B-8 SHAP stability ─────────────────────────────────────────────")
import shap
from sklearn.inspection import permutation_importance
from sklearn.pipeline import Pipeline as SkPipeline

FEATS36 = feature_names(FEAT_FULL())
shap_key = f'{ABL_DIR}/cache/shap_stability_{len(SPLITS)}.pkl'

def _shap_fold(k, tr, te):
    m = clone(make_pipe(FEAT_FULL(), clf='xgb')).fit(X_cl_raw[tr], y_cl[tr])
    prep = SkPipeline(m.steps[:3]); Zte = prep.transform(X_cl_raw[te]); clf = m.named_steps['clf']
    sv = shap.TreeExplainer(clf).shap_values(Zte)
    pi = permutation_importance(clf, Zte, y_cl[te], scoring='roc_auc', n_repeats=10, random_state=SEED + k)
    return k, te, sv, Zte, np.abs(sv).mean(0), pi.importances_mean

if os.path.exists(shap_key):
    shap_out = pickle.load(open(shap_key, 'rb'))
else:
    shap_out = Parallel(n_jobs=N_JOBS)(delayed(_shap_fold)(k, tr, te) for k, (tr, te) in enumerate(SPLITS))
    pickle.dump(shap_out, open(shap_key, 'wb'))

G = np.array([o[4] for o in shap_out])          # folds × 36 mean|SHAP|
PI = np.array([o[5] for o in shap_out])          # folds × 36 permutation importance
rank = pd.DataFrame(G, columns=FEATS36).rank(axis=1, ascending=False)

iu = np.triu_indices(len(G), 1)
rho_ff = stats.spearmanr(G.T).correlation[iu]
top5 = [set(np.argsort(-g)[:5]) for g in G]
jac = [len(top5[i] & top5[j]) / len(top5[i] | top5[j]) for i, j in zip(*iu)]
rho_pi = [stats.spearmanr(g, p).correlation for g, p in zip(G, PI)]

# seed stability: full data, 10 seeds, stochastic subsampling enabled
imp_all = SimpleImputer(strategy='median').fit(X_cl_raw)
Z_all = StandardScaler().fit_transform(ProxyFeatures().fit(None).transform(imp_all.transform(X_cl_raw)))
seed_imp = []
for s in range(10):
    Zs, ys = SMOTE(random_state=s).fit_resample(Z_all, y_cl)
    xm = XGBClassifier(**{**TUNED['xgb'], 'subsample': 0.8}, eval_metric='logloss',
                       random_state=s, verbosity=0, n_jobs=1).fit(Zs, ys)
    seed_imp.append(np.abs(shap.TreeExplainer(xm).shap_values(Z_all)).mean(0))
seed_imp = np.array(seed_imp)
rho_seed = stats.spearmanr(seed_imp.T).correlation[np.triu_indices(10, 1)]

ci = lambda a: f'{np.mean(a):.3f} (2.5–97.5%: {np.percentile(a, 2.5):.3f}–{np.percentile(a, 97.5):.3f})'
stab = pd.DataFrame({'Stability measure': [
    'Spearman ρ of SHAP rankings, fold vs fold (all 36 features)',
    'Jaccard overlap of top-5 SHAP features, fold vs fold',
    'Spearman ρ of SHAP rankings, seed vs seed (full data)',
    'Spearman ρ, SHAP vs permutation importance (same fold)'],
    'Value': [ci(rho_ff), ci(jac), ci(rho_seed), ci(rho_pi)]}).set_index('Stability measure')
display(stab); stab.to_csv(f'{TAB_DIR}/E8_shap_stability.csv')

order = np.argsort(-G.mean(0))
top_tab = pd.DataFrame({
    'feature': [FEATS36[i] for i in order[:15]],
    'block': ['clinical' if i < 13 else 'ECG-proxy' for i in order[:15]],
    'mean |SHAP|': G.mean(0)[order[:15]],
    'median rank (IQR)': [f'{rank.iloc[:, i].median():.0f} ({rank.iloc[:, i].quantile(.25):.0f}–{rank.iloc[:, i].quantile(.75):.0f})'
                          for i in order[:15]],
    '% folds in top-5': [f'{np.mean([i in t for t in top5]):.0%}' for i in order[:15]],
    'perm. importance (ΔAUC)': PI.mean(0)[order[:15]],
}).set_index('feature')
display(top_tab.round(4)); top_tab.to_csv(f'{TAB_DIR}/E8_shap_top_features.csv')

share_proxy = G[:, 13:].sum(1) / G.sum(1)
dup = pd.DataFrame({c: {'clinical copy': G[:, FEATS36.index(c)].mean(),
                        'proxy copy': G[:, FEATS36.index(f'proxy:{c}')].mean()}
                    for c in PROXY_GROUPS['raw']}).T
dup['proxy share of total'] = dup['proxy copy'] / dup.sum(1)
print(f"Share of total |SHAP| carried by the ECG-proxy block: {ci(share_proxy)}")
display(dup.round(4)); dup.to_csv(f'{TAB_DIR}/E8_duplicate_attribution_split.csv')

# Figure A7: rank stability of the top-15 features
fig, ax = plt.subplots(figsize=(9, 5.5))
ax.boxplot([rank.iloc[:, i] for i in order[:15]], vert=False)
ax.set_yticks(range(1, 16)); ax.set_yticklabels([FEATS36[i] for i in order[:15]])
ax.invert_yaxis(); ax.set_xlabel(f'SHAP rank across {len(G)} CV folds (1 = most important)')
ax.set_title('Figure A7: Stability of the SHAP feature ranking', fontweight='bold')
plt.tight_layout(); plt.savefig(f'{FIG_DIR}/figA7_shap_rank_stability.png', dpi=200, bbox_inches='tight'); plt.show()

# Figure A8: dependence plots on pooled out-of-fold SHAP values (repeat 1)
rep0 = [o for o in shap_out if o[0] < N_FOLDS]
SV0 = np.zeros((len(y_cl), len(FEATS36))); Z0 = np.zeros_like(SV0)
for _, te, sv, Zte, *_ in rep0:
    SV0[te], Z0[te] = sv, Zte
raw_imp = imp_all.transform(X_cl_raw)
fig, axes = plt.subplots(1, 5, figsize=(20, 3.8))
for ax, c in zip(axes, ['oldpeak', 'ca', 'thal', 'cp', 'thalach']):
    j = FEATS36.index(c)
    ax.scatter(raw_imp[:, CLIN_COLS.index(c)] + np.random.RandomState(0).uniform(-.08, .08, len(y_cl)),
               SV0[:, j], c=y_cl, cmap='coolwarm', s=14, alpha=.75, edgecolors='none')
    ax.axhline(0, color='k', lw=.7); ax.set_xlabel(c); ax.set_ylabel('SHAP value')
plt.suptitle('Figure A8: SHAP dependence (out-of-fold; red = disease)', fontweight='bold')
plt.tight_layout(); plt.savefig(f'{FIG_DIR}/figA8_shap_dependence.png', dpi=200, bbox_inches='tight'); plt.show()

# Figure A9: local explanations for one TP, FN, FP and TN patient
pred = RESULTS[REF]['oof'][0]          # soft-voting out-of-fold probability, repeat 1
cases = {'True positive': (y_cl == 1) & (pred >= .5), 'False negative': (y_cl == 1) & (pred < .5),
         'False positive': (y_cl == 0) & (pred >= .5), 'True negative': (y_cl == 0) & (pred < .5)}
fig, axes = plt.subplots(1, 4, figsize=(20, 4))
for ax, (lab, msk) in zip(axes, cases.items()):
    idx = np.where(msk)[0]
    if not len(idx):
        ax.set_axis_off(); continue
    i = idx[np.argmax(np.abs(pred[idx] - .5))]
    top = np.argsort(-np.abs(SV0[i]))[:8][::-1]
    ax.barh([FEATS36[t] for t in top], SV0[i, top], color=['#c0392b' if v > 0 else '#2980b9' for v in SV0[i, top]])
    ax.axvline(0, color='k', lw=.7)
    ax.set_title(f'{lab} (patient {i}, p={pred[i]:.2f})', fontweight='bold', fontsize=10)
plt.suptitle('Figure A9: Local SHAP explanations (XGBoost member, out-of-fold)', fontweight='bold')
plt.tight_layout(); plt.savefig(f'{FIG_DIR}/figA9_shap_local.png', dpi=200, bbox_inches='tight'); plt.show()


### 17B-9. Dual-branch network: does cross-attention help?

In [ ]:
# ══════════════════════════════════════════════════════════════════════════
# 17B-9. DUAL-BRANCH NETWORK: IS CROSS-ATTENTION USEFUL? (reviewer row 14)
# MCF-BiModNet (Section IV-G of the paper) reshapes each 32-d branch output to
# a sequence of length 1 before multi-head attention. With one key, softmax
# gives weight exactly 1, so attention reduces to a linear map of the clinical
# branch (W_O·W_V·h_clin + b) and the ECG "query" has no effect. We test:
#   • clinical-only MLP (what Section 14 actually trains)
#   • single MLP on all 36 features
#   • dual branch + concatenation (no attention)
#   • dual branch + cross-attention, sequence length 1 (paper architecture)
#   • dual branch + linear layer in place of attention (theoretical equivalent)
#   • dual branch + token-level cross-attention: every feature is a token, so
#     ECG-proxy tokens attend over 13 clinical tokens (softmax is non-trivial)
# 3 × 10-fold CV, identical preprocessing/SMOTE, fixed training budget.
# ══════════════════════════════════════════════════════════════════════════
print("── 17B-9 Cross-attention ablation ───────────────────────────────────")
DL_SPLITS = SPLITS[:3 * N_FOLDS]
import tensorflow as tf

class FeatureTokens(tf.keras.layers.Layer):
    """Feature tokenizer: token_j = x_j · w_j + b_j  (one learned embedding per feature)."""
    def __init__(self, dim, **kw):
        super().__init__(**kw); self.dim = dim
    def build(self, shape):
        self.w = self.add_weight(shape=(shape[-1], self.dim), initializer='glorot_uniform', name='w')
        self.b = self.add_weight(shape=(shape[-1], self.dim), initializer='zeros', name='b')
    def call(self, x):
        return x[..., None] * self.w + self.b

class DualBranchNet(BaseEstimator, ClassifierMixin):
    def __init__(self, mode='dual_xattn', n_clin=13, epochs=200, batch_size=16, lr=1e-3, seed=SEED):
        self.mode, self.n_clin, self.epochs, self.batch_size, self.lr, self.seed = \
            mode, n_clin, epochs, batch_size, lr, seed

    def _branch(self, x, L):
        x = L.Dense(64, activation='relu')(x); x = L.BatchNormalization()(x)
        x = L.Dropout(0.4)(x); return L.Dense(32, activation='relu')(x)

    def _build(self, n_feat):
        import tensorflow as tf
        L = tf.keras.layers
        tf.keras.utils.set_random_seed(self.seed)
        inp = L.Input(shape=(n_feat,))
        clin = L.Lambda(lambda t: t[:, :self.n_clin])(inp)
        ecg = L.Lambda(lambda t: t[:, self.n_clin:])(inp)
        if self.mode == 'mlp_clinical':
            h = self._branch(clin, L)
        elif self.mode == 'mlp_all':
            h = self._branch(inp, L)
        else:
            hc, he = self._branch(clin, L), self._branch(ecg, L)
            if self.mode == 'dual_concat':
                h = L.Concatenate()([hc, he])
            elif self.mode == 'dual_xattn':           # paper: sequence length 1
                q, kv = L.Reshape((1, 32))(he), L.Reshape((1, 32))(hc)
                att = L.MultiHeadAttention(num_heads=4, key_dim=8)(q, kv)
                h = L.Flatten()(L.Add()([att, q]))
            elif self.mode == 'dual_linear':          # equivalent of the above
                h = L.Add()([L.Dense(32)(hc), he])
            elif self.mode == 'dual_xattn_tokens':    # one token per feature
                ct, et = FeatureTokens(16)(clin), FeatureTokens(16)(ecg)
                att = L.MultiHeadAttention(num_heads=4, key_dim=8)(et, ct)
                h = L.GlobalAveragePooling1D()(L.LayerNormalization()(L.Add()([att, et])))
                h = L.Concatenate()([h, hc])
            else:
                raise ValueError(self.mode)
        h = L.Dense(16, activation='relu')(h)
        out = L.Dense(1, activation='sigmoid')(h)
        model = tf.keras.Model(inp, out)
        model.compile(optimizer=tf.keras.optimizers.Adam(self.lr), loss='binary_crossentropy')
        return model

    def fit(self, X, y):
        import tensorflow as tf
        tf.config.threading.set_intra_op_parallelism_threads(1)
        self.model_ = self._build(X.shape[1])
        cbs = [tf.keras.callbacks.EarlyStopping(patience=15, restore_best_weights=True),
               tf.keras.callbacks.ReduceLROnPlateau(factor=0.5, patience=7, min_lr=1e-6)]
        rng = np.random.RandomState(self.seed); perm = rng.permutation(len(y))   # shuffle before val split
        self.model_.fit(X[perm].astype('float32'), y[perm].astype('float32'), validation_split=0.2,
                        epochs=self.epochs, batch_size=self.batch_size, callbacks=cbs, verbose=0)
        self.classes_ = np.array([0, 1]); return self

    def predict_proba(self, X):
        p = self.model_.predict(X.astype('float32'), verbose=0).ravel()
        return np.column_stack([1 - p, p])

    def predict(self, X):
        return (self.predict_proba(X)[:, 1] >= 0.5).astype(int)

DL_VARIANTS = {
    'DL: clinical-only MLP (Section 14)':        'mlp_clinical',
    'DL: single MLP on all 36 features':         'mlp_all',
    'DL: dual branch + concatenation':           'dual_concat',
    'DL: dual branch + cross-attention (L=1, paper)': 'dual_xattn',
    'DL: dual branch + linear layer (≡ L=1 attention)': 'dual_linear',
    'DL: dual branch + token-level cross-attention':  'dual_xattn_tokens',
}
for name, mode in DL_VARIANTS.items():
    evaluate(name, make_pipe(FEAT_FULL(), clf=DualBranchNet(mode)), splits=DL_SPLITS,
             n_jobs=min(4, os.cpu_count() or 1))

ref_dl = 'DL: dual branch + cross-attention (L=1, paper)'
abl9 = summarize([REF, 'Clinical only'] + list(DL_VARIANTS), ref_dl, title='E9_cross_attention')
display(fmt_table(abl9))

# Numerical check: attention weights with sequence length 1 are exactly 1
mha = tf.keras.layers.MultiHeadAttention(num_heads=4, key_dim=8)
q = tf.random.normal((5, 1, 32)); kv = tf.random.normal((5, 1, 32))
_, w = mha(q, kv, return_attention_scores=True)
q2 = tf.random.normal((5, 1, 32))
out1, out2 = mha(q, kv), mha(q2, kv)
print(f"Attention weights with L=1: min={float(tf.reduce_min(w)):.6f}, max={float(tf.reduce_max(w)):.6f}; "
      f"output change when the ECG query is replaced: {float(tf.reduce_max(tf.abs(out1 - out2))):.2e}")


### 17B-10. PTB-XL ECG branch (D2)
Requires the PTB-XL records (downloaded from PhysioNet if missing; ~25 MB for 1,212 records).

In [ ]:
# ══════════════════════════════════════════════════════════════════════════
# 17B-10. PTB-XL ECG BRANCH (D2) ABLATION
# D2 is the parallel ECG-only benchmark (PTB-XL labels, not Cleveland labels).
# Ablate: lead choice, tsfresh parameter set, 12-lead vs single lead,
# VarianceThreshold fitted inside vs outside the folds, classifier, SMOTE,
# training-set size (303 → 1212 records), and signal-quality subgroups.
# Records are read from data/ptbxl (or PTB_BASE_DIR in this notebook); missing
# ones are downloaded from PhysioNet (PTB-XL v1.0.3, open access).
# ══════════════════════════════════════════════════════════════════════════
print("── 17B-10 PTB-XL ECG-branch ablation ────────────────────────────────")
import ast, urllib.request
import wfdb
from tsfresh import extract_features
from tsfresh.utilities.dataframe_functions import impute as ts_impute
from tsfresh.feature_extraction import EfficientFCParameters, MinimalFCParameters
from sklearn.feature_selection import VarianceThreshold

PTB_DIR = next((d for d in [globals().get('PTB_BASE_DIR'), 'data/ptbxl', '../data/ptbxl']
                if d and os.path.exists(os.path.join(d, 'ptbxl_database.csv'))), 'data/ptbxl')
PTB_URL = 'https://physionet.org/files/ptb-xl/1.0.3/'
os.makedirs(PTB_DIR, exist_ok=True)
for f in ('ptbxl_database.csv', 'scp_statements.csv'):
    if not os.path.exists(f'{PTB_DIR}/{f}'):
        urllib.request.urlretrieve(PTB_URL + f, f'{PTB_DIR}/{f}')

ptb = pd.read_csv(f'{PTB_DIR}/ptbxl_database.csv', index_col='ecg_id')
scp = pd.read_csv(f'{PTB_DIR}/scp_statements.csv', index_col=0)
scp = scp[scp.diagnostic == 1.0]

def _is_normal(raw):                       # same rule as Section 6
    d = {k: v for k, v in ast.literal_eval(raw).items() if k in scp.index and v > 0}
    return bool(d) and list(d) == ['NORM']
ptb['ecg_target'] = ptb.scp_codes.apply(lambda r: 0 if _is_normal(r) else 1)

def ptb_sample(n, seed=SEED):              # proportional stratified sample (Section 6)
    d0, d1 = ptb[ptb.ecg_target == 0], ptb[ptb.ecg_target == 1]
    n0 = int(n * len(d0) / len(ptb))
    return pd.concat([d0.sample(n=n0, random_state=seed),
                      d1.sample(n=n - n0, random_state=seed)]).sample(frac=1, random_state=seed)

N_BIG = int(os.environ.get('MCF_PTB_N', 1212))
s_big = ptb_sample(N_BIG)                  # the 303-record sample is a subset of this one
s_303 = ptb_sample(303)
assert set(s_303.index) <= set(s_big.index)

def read_signals(df):
    sig = []
    for fn in df.filename_lr:
        base = os.path.join(PTB_DIR, fn)
        for ext in ('hea', 'dat'):
            if not os.path.exists(f'{base}.{ext}'):
                os.makedirs(os.path.dirname(base), exist_ok=True)
                urllib.request.urlretrieve(f'{PTB_URL}{fn}.{ext}', f'{base}.{ext}')
        sig.append(wfdb.rdsamp(base)[0])
    return np.stack(sig)                   # (n, 1000, 12)

LEADS = ['I', 'II', 'III', 'aVR', 'aVL', 'aVF', 'V1', 'V2', 'V3', 'V4', 'V5', 'V6']

def tsfresh_lead(signals, ids, lead, params, tag):
    key = f'{ABL_DIR}/cache/tsfresh_{tag}_{lead}_{len(ids)}.pkl'
    if os.path.exists(key):
        return pickle.load(open(key, 'rb'))
    j = LEADS.index(lead)
    long = pd.DataFrame({'id': np.repeat(np.arange(len(ids)), signals.shape[1]),
                         'time': np.tile(np.arange(signals.shape[1]), len(ids)),
                         'value': signals[:, :, j].ravel()})
    F = extract_features(long, column_id='id', column_sort='time', column_value='value',
                         default_fc_parameters=params, impute_function=ts_impute,
                         disable_progressbar=True, n_jobs=os.cpu_count() or 1)
    F.index = ids; F.columns = [f'{lead}__{c}' for c in F.columns]
    pickle.dump(F, open(key, 'wb')); return F

t0 = time.time()
SIG_BIG = read_signals(s_big)
print(f"Loaded {len(SIG_BIG)} PTB-XL records in {time.time() - t0:.0f}s; "
      f"normal={np.sum(s_big.ecg_target == 0)}, abnormal={np.sum(s_big.ecg_target == 1)}")
pos303 = np.array([s_big.index.get_loc(i) for i in s_303.index])

F_eff = {'II': tsfresh_lead(SIG_BIG, s_big.index, 'II', EfficientFCParameters(), 'eff')}
for ld in ('I', 'V1', 'V5'):
    F_eff[ld] = tsfresh_lead(SIG_BIG[pos303], s_303.index, ld, EfficientFCParameters(), 'eff')
F_min = pd.concat([tsfresh_lead(SIG_BIG, s_big.index, ld, MinimalFCParameters(), 'min') for ld in LEADS], axis=1)
y303 = s_303.ecg_target.values
X_eff = {ld: F.loc[s_303.index].values for ld, F in F_eff.items()}
X_min_II = F_min.loc[s_303.index, [c for c in F_min.columns if c.startswith('II__')]].values
X_min_12 = F_min.loc[s_303.index].values
print(f"tsfresh features: Efficient Lead II={X_eff['II'].shape[1]}, Minimal 12-lead={X_min_12.shape[1]}")

ECG_SPLITS = list(RepeatedStratifiedKFold(n_splits=N_FOLDS, n_repeats=N_REPEATS,
                                          random_state=SEED).split(X_eff['II'], y303))

def ecg_pipe(clf='voting', sampler='smote', vt=True):
    steps = [('vt', VarianceThreshold(0.01))] if vt else []
    steps += [('scale', StandardScaler())]
    if sampler:
        steps.append(('sampler', SAMPLERS[sampler]()))
    steps.append(('clf', make_clf(clf)))
    return ImbPipeline(steps)

X_glob_vt = StandardScaler().fit_transform(VarianceThreshold(0.01).fit_transform(X_eff['II']))
ECG_VARIANTS = {
    '[ECG] Lead II, Efficient tsfresh, soft voting (reference)': (ecg_pipe(), X_eff['II']),
    '[ECG] VarianceThreshold + scaling fitted on all records':   (ecg_pipe(vt=False), X_glob_vt),
    '[ECG] Random Forest only (Section 13 best)':                (ecg_pipe('rf'), X_eff['II']),
    '[ECG] No SMOTE':                                            (ecg_pipe(sampler=None), X_eff['II']),
    '[ECG] Minimal tsfresh (10 features), Lead II':              (ecg_pipe(), X_min_II),
    '[ECG] Lead I, Efficient':                                   (ecg_pipe(), X_eff['I']),
    '[ECG] Lead V1, Efficient':                                  (ecg_pipe(), X_eff['V1']),
    '[ECG] Lead V5, Efficient':                                  (ecg_pipe(), X_eff['V5']),
    '[ECG] 12-lead Minimal tsfresh (120 features)':              (ecg_pipe(), X_min_12),
    '[ECG] 12-lead Minimal + Lead II Efficient':                 (ecg_pipe(), np.hstack([X_min_12, X_eff['II']])),
}
for name, (pipe, X) in ECG_VARIANTS.items():
    evaluate(name, pipe, X=X, y=y303, splits=ECG_SPLITS)

_y_backup = y_cl
y_cl = y303                                      # summarize() uses y_cl for calibration slopes
ECG_REF = '[ECG] Lead II, Efficient tsfresh, soft voting (reference)'
abl10 = summarize(list(ECG_VARIANTS), ECG_REF, title='E10_ptbxl_branch')
y_cl = _y_backup
display(fmt_table(abl10))

# ── Training-set size: 303 → 606 → 1212 records (unpaired, own CV splits) ─
size_rows = []
X_big_II = F_eff['II'].values; y_big = s_big.ecg_target.values
for n in (303, 606, N_BIG):
    sub = np.arange(n) if n != 303 else pos303
    Xs, ys = X_big_II[sub], y_big[sub]
    spl = list(RepeatedStratifiedKFold(n_splits=N_FOLDS, n_repeats=3, random_state=SEED).split(Xs, ys))
    r = evaluate(f'[ECG-size] Lead II Efficient, n={n}', ecg_pipe(), X=Xs, y=ys, splits=spl)['folds']
    mu, hw, _ = corrected_t(r.AUC); ma, hwa, _ = corrected_t(r.Accuracy)
    size_rows.append({'Training records': n, 'Unique patients': s_big.iloc[sub].patient_id.nunique(),
                      'AUC [95% CI]': f'{mu:.3f} [{mu - hw:.3f}, {mu + hw:.3f}]',
                      'Accuracy [95% CI]': f'{ma:.3f} [{ma - hwa:.3f}, {ma + hwa:.3f}]'})
size_df = pd.DataFrame(size_rows).set_index('Training records')
display(size_df); size_df.to_csv(f'{TAB_DIR}/E10_ptbxl_sample_size.csv')

# ── Signal-quality subgroups (PTB-XL annotations) on out-of-fold predictions ─
noise_cols = ['baseline_drift', 'static_noise', 'burst_noise', 'electrodes_problems']
noisy = s_303[noise_cols].notna().any(axis=1).values
p_ecg = np.nanmean(RESULTS[ECG_REF]['oof'], axis=0)
qual = pd.DataFrame({
    'n': [len(y303), int((~noisy).sum()), int(noisy.sum())],
    'abnormal': [int(y303.sum()), int(y303[~noisy].sum()), int(y303[noisy].sum())],
    'AUC': [roc_auc_score(y303, p_ecg), roc_auc_score(y303[~noisy], p_ecg[~noisy]),
            roc_auc_score(y303[noisy], p_ecg[noisy])]},
    index=['All 303 records', 'No noise annotation', 'Any noise/electrode annotation'])
display(qual.round(3)); qual.to_csv(f'{TAB_DIR}/E10_ptbxl_signal_quality.csv')

forest_plot(abl10.drop(ECG_REF), 'Figure A10: PTB-XL ECG branch — ΔAUC vs Lead II reference',
            'figA10_ptbxl_branch.png', ref_label='Lead II Efficient')


### 17B-11. Verdict: which claims survive?

In [ ]:
# ══════════════════════════════════════════════════════════════════════════
# 17B-11. VERDICT — the paper's claims re-tested under the controlled protocol
# A claim is "supported" only if the 95% CI of the paired difference excludes
# zero in the claimed direction.
# ══════════════════════════════════════════════════════════════════════════
print("── 17B-11 Claim-by-claim verdict ────────────────────────────────────")

def paired(a, b, m='AUC'):
    fa, fb = RESULTS[a]['folds'], RESULTS[b]['folds']; idx = fa.index.intersection(fb.index)
    d, hw, p = corrected_t(fa.loc[idx, m].values - fb.loc[idx, m].values)
    return d, d - hw, d + hw, p

def verdict(lo, hi, direction=+1):
    if direction > 0:
        return 'Supported' if lo > 0 else ('Contradicted' if hi < 0 else 'Not supported (CI includes 0)')
    return 'Supported' if hi < 0 else ('Contradicted' if lo > 0 else 'Not supported (CI includes 0)')

CLIN = '− ECG-proxy block (clinical only)'
rows = []
def add(claim, paper, a, b, m='AUC', direction=+1, scale=1.0, unit=''):
    d, lo, hi, p = paired(a, b, m)
    rows.append({'Claim in v4 paper': claim, 'Paper value': paper,
                 'Controlled estimate [95% CI]': f'{d * scale:+.3f}{unit} [{lo * scale:+.3f}, {hi * scale:+.3f}]',
                 'p': f'{p:.3f}', 'Verdict': verdict(lo, hi, direction)})

add('ECG-proxy fusion improves accuracy over clinical-only', '+7.04 pp', REF, CLIN, 'Accuracy', +1, 100, ' pp')
add('ECG-proxy fusion improves AUC over clinical-only', '+0.87 pp (0.9535 vs 0.9459)', REF, CLIN, 'AUC')
add('Removing the ECG-proxy layer lowers AUC', '≈ −0.06', CLIN, REF, 'AUC', -1)
add('Removing SMOTE causes the largest drop', '≈ −3.3% accuracy', '− SMOTE (no resampling)', REF, 'Accuracy', -1, 100, ' pp')
add('Removing SMOTE lowers AUC', '"noticeable drop"', '− SMOTE (no resampling)', REF, 'AUC', -1)
add('Ensemble beats a single (XGBoost) classifier', 'AUC −0.014 without ensemble', '− soft voting → XGBoost only', REF, 'AUC', -1)
add('Feature engineering contributes', 'AUC −0.005 without it', 'Clinical + 18 Gaussian-noise features', REF, 'AUC', -1)
add('ECG-proxy block beats an equal-size non-ECG block', '(implied)', REF, 'Clinical + 18 engineered non-ECG features', 'AUC')
if 'DL: dual branch + concatenation' in RESULTS:
    add('Cross-attention improves the dual-branch network', '(implied by MCF-BiModNet)',
        'DL: dual branch + cross-attention (L=1, paper)', 'DL: dual branch + concatenation', 'AUC')
verd = pd.DataFrame(rows).set_index('Claim in v4 paper')

extra = []
if 'rho_ff' in globals():
    extra.append({'Claim in v4 paper': 'SHAP ranking is a stable basis for clinical interpretation',
                  'Paper value': 'thal, ca, cp, oldpeak dominate',
                  'Controlled estimate [95% CI]': f'fold-to-fold Spearman ρ = {np.mean(rho_ff):.2f}; '
                                                  f'top-5 Jaccard = {np.mean(jac):.2f}',
                  'p': '—', 'Verdict': 'Partly supported (top features stable, tail unstable)'})
if 'ext' in globals():
    a = ext.xs('Fused (full system)', level='Model')['AUC']
    extra.append({'Claim in v4 paper': 'Performance generalises beyond Cleveland', 'Paper value': '(not tested)',
                  'Controlled estimate [95% CI]': ', '.join(f'{s} AUC={v:.3f}' for s, v in a.items()),
                  'p': '—', 'Verdict': 'See 17B-7 (lower AUC; fused ≈ clinical)'})
if 'ho' in globals():
    extra.append({'Claim in v4 paper': 'Hold-out accuracy 88.52% is representative', 'Paper value': '0.8852 (seed 42)',
                  'Controlled estimate [95% CI]': f'{ho.fused_Acc.mean():.3f} over {len(ho)} splits '
                                                  f'(seed 42 = {stats.percentileofscore(ho.fused_Acc, s42.fused_Acc):.0f}th pct)',
                  'p': '—', 'Verdict': 'Optimistic single split'})
verd = pd.concat([verd, pd.DataFrame(extra).set_index('Claim in v4 paper')]) if extra else verd
display(verd); verd.to_csv(f'{TAB_DIR}/E11_verdict.csv')

# Master table of every variant evaluated in this section
master = []
for n, r in RESULTS.items():
    f = r['folds']; row = {'Variant': n, 'folds': len(f)}
    for m in ('AUC', 'Accuracy', 'Sensitivity', 'Specificity', 'F1', 'Brier'):
        mu, hw, _ = corrected_t(f[m]); row[m] = mu; row[f'{m}_ci'] = hw
    master.append(row)
pd.DataFrame(master).set_index('Variant').to_csv(f'{TAB_DIR}/ALL_variants.csv')
print(f"\n{len(master)} variants evaluated; tables in {TAB_DIR}, figures in {FIG_DIR}")
